In [1]:
import torch 
import math
from torch import nn

In [2]:
class InputEmbedding(nn.Module):
    def __init__(self,vocab_size:int ,d_model:int)->None:
        super().__init__()
        self.vocab_size = vocab_size
        self.d_model = d_model
        
        self.embedding = nn.Embedding(vocab_size,d_model)
        print(self.embedding.weight)
    def forward(self, x):
        return self.embedding(x)
    
    

In [15]:
batch_size = 3
vocab_size = 12
d_model = 4

# token ids to fetch embeddings
x = torch.tensor([[0,1,2],
                  [3,4,5],
                  [7,8,9]])

print(x.shape)   # (2,3)

# Creating embedding table of shape (vocab_size, d_model)
# Here -> (6,4)
input_emb = InputEmbedding(vocab_size,d_model)

# token ids shape: (batch_size, seq_len)
# (2,3)

# after embedding lookup:
# (batch_size, seq_len) --> (batch_size, seq_len, d_model)
# (2,3) --> (2,3,4)

output = input_emb(x)

print(output)
print(output.shape)

torch.Size([3, 3])
Parameter containing:
tensor([[-0.5432, -0.5206, -0.5643,  1.5364],
        [ 0.1985,  0.6983,  0.8900,  1.6043],
        [ 1.5261, -0.3656,  1.7865, -2.5328],
        [ 0.2268, -1.0703, -0.3531,  1.8218],
        [-1.2131, -0.4753, -0.4226,  0.4243],
        [-0.0370, -1.7513,  0.4565, -0.4480],
        [-0.3550, -0.2673, -0.4594, -0.0058],
        [-0.7986, -0.2013,  0.5208,  0.7939],
        [-0.4714,  0.8717, -1.5561,  0.5297],
        [ 0.6042,  1.4465, -0.2939,  0.5461],
        [-0.3657,  0.6850,  0.4066,  1.6411],
        [-0.8892, -0.1849,  0.8257, -0.1077]], requires_grad=True)
tensor([[[-0.5432, -0.5206, -0.5643,  1.5364],
         [ 0.1985,  0.6983,  0.8900,  1.6043],
         [ 1.5261, -0.3656,  1.7865, -2.5328]],

        [[ 0.2268, -1.0703, -0.3531,  1.8218],
         [-1.2131, -0.4753, -0.4226,  0.4243],
         [-0.0370, -1.7513,  0.4565, -0.4480]],

        [[-0.7986, -0.2013,  0.5208,  0.7939],
         [-0.4714,  0.8717, -1.5561,  0.5297],
      

In [16]:
class PositionalEncoding(nn.Module):

    def __init__(self, seq_len, d_model):

        super().__init__()

        self.seq_len = seq_len
        self.d_model = d_model
        # Creating empty positional encoding matrix
        # Shape: (seq_len, d_model)
        # rows    -> positions of tokens
        # columns -> embedding dimensions
        self.pe = torch.zeros((self.seq_len, self.d_model))
        # Creating position vector
        # Example for seq_len=3:
        # [0,1,2] -> [[0],[1],[2]]
        # Shape: (seq_len,1)
        # Each row represents token position
        position = torch.arange(0, seq_len).unsqueeze(1)
        # Creating frequency/divisor terms for even dimensions
        # arange(0,d_model,2) gives:
        # [0,2,4,6...]
        # Different dimensions get different frequencies:
        # smaller dimensions -> faster oscillation
        # larger dimensions  -> slower oscillation
        # Shape: (d_model/2,)
        div_term = torch.exp(torch.arange(0, d_model, 2).float()* (-math.log(10000.0) / d_model))
        # Broadcasting:
        # position shape  -> (seq_len,1)
        # div_term shape  -> (d_model/2,)
        # Result shape becomes:
        # (seq_len,d_model/2)
        # Fill even columns (0,2,4...) using sine
        self.pe[:, ::2] = torch.sin(position * div_term)
        # Fill odd columns (1,3,5...) using cosine
        self.pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('positional_encoding',self.pe)
        print(self.pe)
    def forward(self,x):
            return x+self.pe[:x.shape[1]]
        

In [17]:
positional_encoding = PositionalEncoding(3,4)
output=positional_encoding(output)
print(output.shape)

tensor([[ 0.0000,  1.0000,  0.0000,  1.0000],
        [ 0.8415,  0.5403,  0.0100,  0.9999],
        [ 0.9093, -0.4161,  0.0200,  0.9998]])
torch.Size([3, 3, 4])


In [18]:
class SelfAttention(nn.Module):

    def __init__(self, d_model: int):

        super().__init__()

        # Number of attention heads
        self.h = 2

        # Total embedding dimension
        self.d_model = d_model

        # d_model must be divisible by number of heads
        assert self.d_model % self.h == 0, "not divisible by number of heads"

        # Dimension handled by each head
        # Example:
        # d_model=8 , h=2 -> d_k=4
        self.d_k = self.d_model // self.h

        # Query projection layer
        # Input shape  : (..., d_model)
        # Output shape : (..., d_model)
        self.w_q = nn.Linear(self.d_model, self.d_model)

        # Key projection layer
        self.w_k = nn.Linear(self.d_model, self.d_model)

        # Value projection layer
        self.w_v = nn.Linear(self.d_model, self.d_model)

    def forward(self, x):
        # x shape:
        # (batch_size, seq_len, d_model)
        # Example:
        # (2,3,8)
        # ---------------------------------------------------
        # CREATE QUERY, KEY, VALUE MATRICES
        # ---------------------------------------------------
        # query shape:
        # (batch_size, seq_len, d_model)
        # (2,3,8)
        query = self.w_q(x)

        # key shape:
        # (batch_size, seq_len, d_model)
        # (2,3,8)
        key = self.w_k(x)

        # value shape:
        # (batch_size, seq_len, d_model)
        # (2,3,8)
        value = self.w_v(x)

        # ---------------------------------------------------
        # SPLIT d_model INTO MULTIPLE HEADS
        # ---------------------------------------------------

        # Before reshape:
        # (batch_size, seq_len, d_model)
        # (2,3,8)

        # After reshape:
        # (batch_size, seq_len, heads, d_k)
        # (2,3,2,4)

        query = query.reshape(
            x.shape[0],
            x.shape[1],
            self.h,
            self.d_k
        )

        key = key.reshape(
            x.shape[0],
            x.shape[1],
            self.h,
            self.d_k
        )

        value = value.reshape(
            x.shape[0],
            x.shape[1],
            self.h,
            self.d_k
        )

        # ---------------------------------------------------
        # MOVE HEAD DIMENSION BEFORE seq_len
        # ---------------------------------------------------

        # Before transpose:
        # (batch_size, seq_len, heads, d_k)
        # (2,3,2,4)

        # After transpose:
        # (batch_size, heads, seq_len, d_k)
        # (2,2,3,4)

        query = query.transpose(-3, -2)

        key = key.transpose(-3, -2)

        value = value.transpose(-3, -2)

        # ---------------------------------------------------
        # CALCULATE ATTENTION SCORES
        # ---------------------------------------------------

        # key.transpose(-2,-1) shape:
        # (batch_size, heads, d_k, seq_len)
        # (2,2,4,3)

        # query shape:
        # (batch_size, heads, seq_len, d_k)
        # (2,2,3,4)

        # attention_score shape:
        # (batch_size, heads, seq_len, seq_len)
        # (2,2,3,3)

        attention_score = query @ key.transpose(-2, -1)

        # ---------------------------------------------------
        # SCALE ATTENTION SCORES
        # ---------------------------------------------------

        # Shape remains:
        # (batch_size, heads, seq_len, seq_len)
        # (2,2,3,3)

        attention_score = attention_score / math.sqrt(self.d_k)

        # ---------------------------------------------------
        # APPLY SOFTMAX
        # ---------------------------------------------------

        # Softmax applied across last dimension
        # Each row becomes probability distribution

        # Shape remains:
        # (batch_size, heads, seq_len, seq_len)
        # (2,2,3,3)

        attention_score = torch.softmax(attention_score, dim=-1)

        # ---------------------------------------------------
        # MULTIPLY WITH VALUE MATRIX
        # ---------------------------------------------------

        # attention_score shape:
        # (2,2,3,3)

        # value shape:
        # (2,2,3,4)

        # Output shape:
        # (batch_size, heads, seq_len, d_k)
        # (2,2,3,4)

        attention_score = attention_score @ value

        # ---------------------------------------------------
        # MOVE seq_len BEFORE HEADS AGAIN
        # ---------------------------------------------------

        # Before transpose:
        # (2,2,3,4)

        # After transpose:
        # (2,3,2,4)

        attention_score = attention_score.transpose(-3, -2)

        # ---------------------------------------------------
        # MERGE ALL HEADS BACK
        # ---------------------------------------------------

        # Before reshape:
        # (batch_size, seq_len, heads, d_k)
        # (2,3,2,4)

        # After reshape:
        # (batch_size, seq_len, d_model)
        # (2,3,8)

        attention_score = attention_score.reshape(
            x.shape[0],
            x.shape[1],
            x.shape[2]
        )

        # Final contextual token representation
        # Shape:
        # (batch_size, seq_len, d_model)

        return attention_score

In [35]:
class LayerNorm(nn.Module):
    def __init__(self,eps=10**(-9)):
        super().__init__()
        
        self.eps = eps
        
    def forward(self,x,attention_score):
        self.add = x+attention_score
        self.mean = torch.mean(self.add,dim=-1).unsqueeze(2)
        self.var = torch.var((self.add-self.mean),dim=-1).unsqueeze(2)
        self.normalize =(self.add-self.mean)/torch.sqrt(self.var+self.eps)
        return self.normalize

In [37]:
class FeefForwardNetwork(nn.Module):
    def __init__(self,d_model):
        super().__init__()
        self.d_model = d_model
        self.layer = nn.Sequential(
            nn.Linear(self.d_model,2048),
            nn.ReLU(),
            nn.Linear(2048,self.d_model)
        )
    def forward(self,x):
        return self.layer(x)

In [38]:
class EncoderBLock(nn.Module):
    def __init__(self,attention,LayerNorm,fnn):
        super().__init__()
        self.attention = attention
        self.LayerNorm = LayerNorm
        self.fnn = fnn
    def forward(self,x):
        attention_output = self.attention(x)
        x= self.LayerNorm(x,attention_output)
        fnn_output = self.fnn(x)
        x=self.LayerNorm(x,fnn_output)
        return x

In [ ]:
attention = SelfAttention(4)

add_norm = LayerNorm()

fnn = FeefForwardNetwork(4)

encoder_output = EncoderBLock(
    attention,
    add_norm,
    fnn
)


In [41]:
encoder_output(output)

tensor([[[-0.3898, -0.5479,  1.4956, -0.5578],
         [-0.4588, -0.5733,  1.4979, -0.4659],
         [-0.3970, -0.4554,  1.4920, -0.6395]],

        [[-0.3633, -0.5904,  1.4929, -0.5392],
         [-0.3244, -0.5631,  1.4887, -0.6012],
         [-0.3616, -0.4038,  1.4812, -0.7158]],

        [[-0.4048, -0.5560,  1.4966, -0.5358],
         [-0.4426, -0.4714,  1.4972, -0.5833],
         [-0.4549, -0.5061,  1.4991, -0.5381]]], grad_fn=<DivBackward0>)